In [ ]:
import streamlit as st
import google.generativeai as genai

# --- PAGE SETUP ---
st.set_page_config(page_title="My VGU Chatbot", page_icon="🤖")

st.title("🤖 AI Chatbot Integration")
st.caption("Powered by Gemini & Streamlit | Created by [Your Name]")

# --- SIDEBAR: API KEY CONFIGURATION ---
with st.sidebar:
    st.header("Settings")
    # We use a password input so the key isn't visible
    api_key = st.text_input("Enter Google API Key", type="password")
    st.markdown("[Get your API key here](https://aistudio.google.com/app/apikey)")

# --- GEMINI CONFIGURATION ---
if api_key:
    try:
        genai.configure(api_key=api_key)
        # Using gemini-1.5-flash as it is fast and free-tier friendly
        # (Note: 'gemini-2.5-flash' in your screenshot doesn't exist yet!)
        model = genai.GenerativeModel("gemini-1.5-flash")
    except Exception as e:
        st.error(f"Error configuring API: {e}")

# --- SESSION STATE (The Memory) ---
# This is crucial! It stops the chat from disappearing when the script reruns.
if "messages" not in st.session_state:
    st.session_state.messages = []

# --- DISPLAY CHAT HISTORY ---
# We loop through the history stored in session_state and display it
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# --- CHAT INPUT HANDLING ---
if prompt := st.chat_input("What is on your mind?"):
    # 1. Check if API key is present
    if not api_key:
        st.warning("Please enter your API Key in the sidebar first!")
        st.stop()

    # 2. Display user message immediately
    with st.chat_message("user"):
        st.markdown(prompt)
    
    # 3. Add user message to history
    st.session_state.messages.append({"role": "user", "content": prompt})

    # 4. Generate Response
    try:
        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                # Generate content using the Gemini model
                response = model.generate_content(prompt)
                st.markdown(response.text)
        
        # 5. Add assistant response to history
        st.session_state.messages.append({"role": "assistant", "content": response.text})
        
    except Exception as e:
        st.error(f"An error occurred: {e}")